# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-11-22 20:47:46--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 48.202.10.80
Connecting to github.com (github.com)|48.202.10.80|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-11-22 20:47:46--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  --.-KB/s    in 0.1s    

2026-11-22 20:47:46 (127 MB/s) - ‘enron_spam

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [6]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [9]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [10]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [11]:
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [19]:
# ensure all cells are strings before combining full email contents per row
def clean_cell(cell):
  return "" if pd.isna(cell) else str(cell)

# format each row to contain the date, subject, message
message_id = [clean_cell(val) for val in enron_spam_data['Message ID']]
subject    = [clean_cell(val) for val in enron_spam_data['Subject']]
message    = [clean_cell(val) for val in enron_spam_data['Message']]
date       = [clean_cell(val) for val in enron_spam_data['Date']]

full_email = [
    f"Date: {d}\n\nSubject: {s}\n\nMessage: {m}"
    for d, s, m in zip(date, subject, message)
]

# store embedded emails with a key of the email's Message ID
email_embeddings = {}

for msg_id, email in zip(message_id, full_email):
  email_embeddings[msg_id] = embed_text(email)


Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [24]:
import json
import numpy as np

# round vector to 3 decimal places and flatten to 1D
def round_1D_json_vec(emb, decimals=3):
  vec         = np.asarray(emb, dtype=float).flatten()
  rounded_vec = [round(i, decimals) for i in vec]
  return json.dumps(rounded_vec)

emails = [
    {
        "Message ID": msg_id,
        "embedding_vector_json": round_1D_json_vec(emb, decimals=3)
    }
    for msg_id, emb in email_embeddings.items()
]

embeddings = pd.DataFrame(emails)
embeddings.head()

,Message ID,embedding_vector_json
0,0,"[-0.374, -0.493, -0.164, -0.197, -0.326, -0.37..."
1,1,"[-0.645, 0.037, 0.215, 0.116, 0.22, -0.184, 0...."
2,2,"[-0.603, -0.475, -0.426, -0.409, 0.083, -0.537..."
3,3,"[-0.716, -0.333, -0.159, -0.128, -0.486, -0.33..."
4,4,"[-0.558, -0.189, 0.384, -0.198, -0.446, -0.021..."


In [25]:
embeddings.to_csv('embeddings.tsv.gz', sep='\t', index=False)

Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [30]:
embeddings = pd.read_csv('embeddings.tsv.gz', sep='\t')

msg_id_with_target = enron_spam_data[["Message ID", "Spam/Ham"]]

# Merge target with embeddings based on Message ID and convert Spam/Ham to 1/0
embeddings_with_target = embeddings.merge(msg_id_with_target, on="Message ID", how="left")
embeddings_with_target['Spam/Ham'] = embeddings_with_target['Spam/Ham'].map({"ham": 0, "spam": 1})
embeddings_with_target

,Message ID,embedding_vector_json,Spam/Ham
0,0,"[-0.374, -0.493, -0.164, -0.197, -0.326, -0.37...",0
1,1,"[-0.645, 0.037, 0.215, 0.116, 0.22, -0.184, 0....",0
2,2,"[-0.603, -0.475, -0.426, -0.409, 0.083, -0.537...",0
3,3,"[-0.716, -0.333, -0.159, -0.128, -0.486, -0.33...",0
4,4,"[-0.558, -0.189, 0.384, -0.198, -0.446, -0.021...",0
...,...,...,...
33711,33711,"[0.068, -0.107, 0.342, -0.028, -0.376, -0.625,...",1
33712,33712,"[0.022, -0.057, 0.25, 0.1, 0.07, -0.42, 0.339,...",1
33713,33713,"[0.097, -0.372, 0.086, -0.094, -0.407, -0.283,...",1
33714,33714,"[0.006, -0.307, -0.044, -0.25, -0.126, -0.508,...",1


In [31]:
embeddings_with_target['Spam/Ham'].value_counts()

,count
Spam/Ham,
1,17171
0,16545


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = np.vstack([json.loads(vec_json) for vec_json in embeddings_with_target['embedding_vector_json']])
y = embeddings_with_target['Spam/Ham']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)

lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, y_train)

y_pred = lin_reg_model.predict(X_test)
predictions = y_pred >= 0.5
accuracy = (y_test == predictions).mean()

coef_df = pd.DataFrame({
    "dim": np.arange(len(lin_reg_model.coef_)),
    "coefficient": lin_reg_model.coef_
})

display(coef_df)
print(f"Accuracy: {accuracy:.4}")

,dim,coefficient
0,0,0.025855
1,1,-0.029435
2,2,0.053819
3,3,0.064585
4,4,0.114119
...,...,...
763,763,-0.024041
764,764,0.034070
765,765,0.093611
766,766,-0.025014


Accuracy: 0.9843


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [36]:
coef_df.to_csv('coefficients.tsv', sep='\t', index=False)

Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [37]:
queries = pd.read_csv('queries.tsv', sep='\t')
queries

,query_id,query
0,1,accounting arrangements
1,2,sales higher than production
2,3,asked lawyer to write letter about unexpected ...
3,4,engineering exam
4,5,discounted tickets
5,6,unexecuted agreement
6,7,well bore
7,8,capacity problem
8,9,london partner
9,10,dormant account


In [39]:
# get Message IDs and matrix of email embeddings
msg_ids = list(email_embeddings.keys())
emails  = np.vstack([np.asarray(emb) for emb in email_embeddings.values()])

# format each row to contain the date, subject, message
query_id = [clean_cell(val) for val in queries['query_id']]
query    = [clean_cell(val) for val in queries['query']]

# store embedded queries with a key of the query's Query ID
query_embeddings = {}

for q_id, query in zip(query_id, query):
  query_embeddings[q_id] = embed_text(query)

# find closest match for email and query
query_results = []

for q_id, q_emb in query_embeddings.items():
  q_vec = np.asarray(q_emb).flatten()

  # calculate distance per email (email matrix where each row is a unique email embedding)
  l2_distances = np.linalg.norm(emails - q_vec, axis=1)

  # find closest email match based on query vector
  nearest_idx    = int(np.argmin(l2_distances))
  nearest_msg_id = msg_ids[nearest_idx]

  query_results.append({
      "query_id": q_id,
      "query_vector_json": round_1D_json_vec(q_vec, decimals=3),
      "Message ID": nearest_msg_id
  })

query_matches = pd.DataFrame(query_results)
query_matches

,query_id,query_vector_json,Message ID
0,1,"[-0.067, 0.486, -0.697, 0.012, 0.353, 0.001, -...",3018
1,2,"[-0.865, -0.246, 0.211, -0.16, -0.095, 0.163, ...",2758
2,3,"[-0.298, -0.203, -0.198, 0.104, -0.608, -0.122...",18071
3,4,"[-0.215, 0.199, -0.022, 0.005, 0.379, -0.211, ...",4594
4,5,"[-0.196, -0.049, -0.286, 0.084, 0.672, -0.281,...",16320
5,6,"[-0.542, -0.275, -0.15, -0.028, 0.283, -0.127,...",20339
6,7,"[-0.124, 0.354, 0.073, -0.094, -0.022, -0.217,...",20281
7,8,"[-0.258, 0.056, -0.195, 0.072, -0.175, 0.306, ...",4349
8,9,"[-0.32, 0.158, -0.184, -0.192, 0.056, 0.259, 0...",31366
9,10,"[-0.298, -0.306, -0.281, -0.208, 0.501, 0.198,...",3894


Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [40]:
query_matches.to_csv('query-matches.tsv', sep='\t', index=False)

Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

none